Author: Krish

In [15]:
# Importing relevant libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime

import warnings
warnings.filterwarnings("ignore")

### User input required
Put the data path on your system in the cell below

In [16]:
data_path = "/Users/varunpopli/Desktop/Data/CAR_-_EP_Flow_Activity_Queue__Agent_Names"

### User input ends

### Reading all filenames in the data folder

In [17]:
folder = Path(data_path)
files = sorted(list(folder.glob("*.csv")) + list(folder.glob("*.xlsx")))
df_main = pd.DataFrame(columns=['Contact Session ID', 'EP Name', 'Flow Name', 'Activity Name', 'Activity Start Timestamp', 
                                'Queue Name', 'Agent Name', 'Termination Reason'])
df_main

,Contact Session ID,EP Name,Flow Name,Activity Name,Activity Start Timestamp,Queue Name,Agent Name,Termination Reason


### Reading all data files
The code chunk below reads and appends all the CAR data files. The first two rows of each file are blank and thus ignored.

In [18]:
i=0
for f in files:
    i = i + 1
    if f.suffix.lower() == ".csv":
        df = pd.read_csv(f, header=2, dtype=str, engine="python", skip_blank_lines=False)
    else:  # .xlsx
        df = pd.read_excel(f, sheet_name=0, header=2, dtype=str)
    df_main = pd.concat([df_main, df], ignore_index=True)
    print(i, f.stem, df.shape)

1 CAR - EP, Flow, Activity, Queue, & Agent Names (01-12-25 - 01-18-25) (52010, 7)
2 CAR - EP, Flow, Activity, Queue, & Agent Names (01-19-25 - 02-01-25) (95495, 7)
3 CAR - EP, Flow, Activity, Queue, & Agent Names (02-02-25 - 02-15-25) (90056, 7)
4 CAR - EP, Flow, Activity, Queue, & Agent Names (02-16-25 - 03-01-25) (88186, 7)
5 CAR - EP, Flow, Activity, Queue, & Agent Names (03-02-25 - 03-15-25) (86377, 7)
6 CAR - EP, Flow, Activity, Queue, & Agent Names (04-07-24 - 04-20-24) (88766, 7)
7 CAR - EP, Flow, Activity, Queue, & Agent Names (04-21-24 - 05-04-24) (89643, 7)
8 CAR - EP, Flow, Activity, Queue, & Agent Names (05-05-24 - 05-18-24) (82575, 7)
9 CAR - EP, Flow, Activity, Queue, & Agent Names (05-19-24 - 06-01-24) (71103, 7)
10 CAR - EP, Flow, Activity, Queue, & Agent Names (06-02-24 - 06-15-24) (84354, 7)
11 CAR - EP, Flow, Activity, Queue, & Agent Names (06-16-24 - 06-29-24) (82124, 7)
12 CAR - EP, Flow, Activity, Queue, & Agent Names (06-30-24 - 07-13-24) (79752, 7)
13 CAR - EP, 

### Time datatype conversion
The code chunk below converts time from string to datetime datatype.

In [19]:
df_main["Activity Start Timestamp"] = df_main["Activity Start Timestamp"].apply(
    lambda x: datetime.strptime(x, "%Y/%m/%d %I:%M:%S %p"))

In [21]:
df_main.head()

,Contact Session ID,EP Name,Flow Name,Activity Name,Activity Start Timestamp,Queue Name,Agent Name,Termination Reason
0,001a3748-8d50-4550-8461-33547983deb0,Main Number Telephony EP,NaN,NaN,2025-01-14 12:39:32,NaN,NaN,NaN
1,001a3748-8d50-4550-8461-33547983deb0,NaN,LACMain,NaN,2025-01-14 12:39:32,NaN,NaN,NaN
2,001a3748-8d50-4550-8461-33547983deb0,Main Number Telephony EP,NaN,LanguageSelectionMenu,2025-01-14 12:39:32,NaN,NaN,NaN
3,001a3748-8d50-4550-8461-33547983deb0,Main Number Telephony EP,LACMain,NaN,2025-01-14 12:39:32,NaN,NaN,NaN
4,001a3748-8d50-4550-8461-33547983deb0,Main Number Telephony EP,NaN,MainMenu,2025-01-14 12:39:45,NaN,NaN,NaN


In [24]:
# --- Step 3: Group by Contact Session ID and aggregate ---
grouped = (
    df_main.groupby('Contact Session ID')
    .agg(
        Call_Start_Time=('Activity Start Timestamp', 'min'),   # earliest timestamp per call
        Count=('Activity Start Timestamp', lambda x: len(set(x))),  # number of unique activity timestamps
        Call_Duration=('Activity Start Timestamp', 
                       lambda x: (max(x) - min(x)).total_seconds() / 60 if len(x) > 1 else 0)
    )
    .reset_index()
)

# --- Step 4: Add columns about day specifically  ---
grouped['DayOfWeekNum'] = grouped['Call_Start_Time'].dt.dayofweek + 1     # 1 = Monday, 7 = Sunday
grouped['Call_Start_Date'] = grouped['Call_Start_Time'].dt.date            # date only (no time)
grouped['Year'] = grouped['Call_Start_Time'].dt.year 
grouped['Month'] = grouped['Call_Start_Time'].dt.month_name() 

# --- Step 5: Save for Power BI ---
grouped.to_csv("combined_calls_transformed_simple.csv", index=False)

display(grouped.head())

,Contact Session ID,Call_Start_Time,Count,Call_Duration,DayOfWeekNum,Call_Start_Date,Year,Month
0,00002422-f51f-458b-82d6-cfa5a3f36fd9,2025-03-13 12:51:21,4,0.900000,4,2025-03-13,2025,March
1,0000a8d5-cecb-46b1-82cf-b7ce07d85b24,2025-03-17 16:53:15,5,0.933333,1,2025-03-17,2025,March
2,00011655-35de-476f-9a8c-dd48ed4d914a,2024-11-06 14:39:01,10,4.133333,3,2024-11-06,2024,November
3,00014a58-a6ce-4cb2-a529-d55e2c9c304d,2025-02-28 08:33:40,8,2.283333,5,2025-02-28,2025,February
4,00015327-f646-462f-a585-0552331eed4e,2025-06-03 07:43:56,3,0.200000,2,2025-06-03,2025,June
